In [ ]:
#!/usr/bin/env python
from scipy.stats import expon
from scipy.stats import uniform
from scipy.stats import norm
from scipy.stats import multivariate_normal
from numpy.random import multinomial
from numpy.random import uniform
import numpy as np
import matplotlib.pyplot as plt
import scipy 
from scipy.stats import truncnorm
import math
from scipy.stats import mvn
np.set_printoptions(suppress=True)
import pyreadr
import time
import sys
import os
from math import comb
from numpy.linalg import LinAlgError
import random
from scipy.linalg import cho_solve, cho_factor
import math
from math import lgamma, exp,comb
np.random.seed(123)



# Fixed hyperparameters

alpha = 0.1
beta = 0.1



data = pyreadr.read_r('archive/cell_lineage/UGPMA100new.rda')
points_inhomo = np.array(data["bt_adj"]).squeeze()
T=25
print(points_inhomo)

ntip=100






N = ntip - 1
K= ntip  # counts of integrals
Ngrid = 100
x4 = np.linspace(0, T, Ngrid + 1)[1:]
xxx=np.concatenate([x4, points_inhomo])

N = ntip - 1
Ngrid = 100
K= ntip  # counts of integrals
x4 = np.linspace(0, T, Ngrid + 1)[1:]
xxx=np.concatenate([x4, points_inhomo])


days = np.concatenate([[0], points_inhomo,[T]])
diff = np.diff(days)
Nfinal = Ngrid + N + K
g_mk = 3 + np.zeros(Nfinal)
g_mk[ Ngrid+N:Ngrid+N+K ]=3*T*diff/sum(diff)
nsim1=1000000
nsim2=1000000


# Build covariance matrix with stable kernel
def expo_quad_kernel(xn, xm):
    return min(xn, xm)

def expo_quad_kernel2(xn, t1, t2):
    # Numerically stable Brownian point-integral covariance
    xn, t1, t2 = float(xn), float(t1), float(t2)
    h = t2 - t1

    if xn <= t1:
        return xn * h
    elif xn < t2:
        u = xn - t1
        return t1 * h + u * h - 0.5 * u**2
    else:  # xn >= t2
        return h * (t1 + 0.5 * h)

def expo_quad_kernel3(t1, t2, t3, t4):
    # Numerically stable Brownian integral-integral covariance
    t1, t2, t3, t4 = float(t1), float(t2), float(t3), float(t4)
    h1 = t2 - t1
    h2 = t4 - t3

    if t1 == t3 and t2 == t4:
        # Var(integral_{t1}^{t2} B(s) ds)
        return t1 * h1**2 + h1**3 / 3.0
    else:
        # In this code the intervals are ordered because i <= j, so t2 <= t3
        return h1 * h2 * (t1 + 0.5 * h1)





def a_coeffs_kmax(kmax):
    if kmax == 1 or kmax == 2:
        return np.array([1.0])

    a_prev = np.array([1.0])  # k = 2

    for k in range(3, kmax + 1):
        Mk = comb(k - 1, 2)
        Mk_prev = comb(k - 2, 2)

        a_cur = np.zeros(Mk + 1, dtype=float)

        for i in range(0, Mk + 1):
            # R index in R was i + 1
            # Python index is i

            if i == 0:
                a_cur[i] = 1.0
                continue

            a_im1_k = a_cur[i - 1]  # a_{i-1,k}

            if i <= Mk_prev:
                a_i_km1 = a_prev[i]  # a_{i,k-1}
            else:
                a_i_km1 = 0.0

            numerator = (
                (comb(k - 1, 2) - i + 1) * a_im1_k
                + comb(k, 2) * a_i_km1
            )

            denominator = comb(k, 2) - i

            a_cur[i] = numerator / denominator

        a_prev = a_cur

    return a_prev


a = a_coeffs_kmax(ntip)

def rhs_value(x, a=a):
    poly = 0.0

    # Equivalent to: for (i in length(a):1)
    for coeff in reversed(a):
       
        poly = poly * x + coeff

    return poly




n_vec = np.arange(ntip, 1, -1)
com_vec = n_vec * (n_vec - 1) / 2


def chol_sample(mean, cov_chol):
    return mean + cov_chol @ np.random.standard_normal(mean.size)


def log_lik(f,ns):
    if np.any(f <= 0):
        return -np.inf
    term1 = np.sum(np.log(f[:, Ngrid:Ngrid+ns]))
    term2 = -np.sum(com_vec * f[:, Ngrid+ns:Nfinal-1])
    Lambda=np.sum(f[:, (Ngrid+ns):Nfinal]) 
    x=exp(-Lambda)
    val=rhs_value( x) 
    logboundprob=((ntip-1)*np.log(1-x)+np.log(val))
    term3=-logboundprob
    
    return term1+term2+term3
    

        

    
def elliptical_slice(initial_theta,prior,lnpdf,pdf_params=(),
                     cur_lnpdf=None,angle_range=None):
    """
    NAME:
       elliptical_slice
    PURPOSE:
       Markov chain update for a distribution with a Gaussian "prior" factored out
    INPUT:
       initial_theta - initial vector
       prior - cholesky decomposition of the covariance matrix 
               (like what numpy.linalg.cholesky returns), 
               or a sample from the prior
       lnpdf - function evaluating the log of the pdf to be sampled
       pdf_params= parameters to pass to the pdf
       cur_lnpdf= value of lnpdf at initial_theta (optional)
       angle_range= Default 0: explore whole ellipse with break point at
                    first rejection. Set in (0,2*pi] to explore a bracket of
                    the specified width centred uniformly at random.
    OUTPUT:
       new_theta, new_lnpdf
    HISTORY:
       Originally written in matlab by Iain Murray (http://homepages.inf.ed.ac.uk/imurray2/pub/10ess/elliptical_slice.m)
       2012-02-24 - Written - Bovy (IAS)
    """
    D= len(initial_theta)
    if cur_lnpdf is None:
        cur_lnpdf= lnpdf(initial_theta,*pdf_params)

    # Set up the ellipse and the slice threshold
    if len(prior.shape) == 1: #prior = prior sample
        nu= prior
    else: #prior = cholesky decomp
        if not prior.shape[0] == D or not prior.shape[1] == D:
            raise IOError("Prior must be given by a D-element sample or DxD chol(Sigma)")
        nu= np.dot(prior,np.random.normal(size=D))
    hh = math.log(np.random.uniform()) + cur_lnpdf

    # Set up a bracket of angles and pick a first proposal.
    # "phi = (theta'-theta)" is a change in angle.
    if angle_range is None or angle_range == 0.:
        # Bracket whole ellipse with both edges at first proposed point
        phi= np.random.uniform()*2.*math.pi
        phi_min= phi-2.*math.pi
        phi_max= phi
    else:
        # Randomly center bracket on current point
        phi_min= -angle_range*np.random.uniform()
        phi_max= phi_min + angle_range
        phi= np.random.uniform()*(phi_max-phi_min)+phi_min

    # Slice sampling loop
    while True:
        # Compute xx for proposed angle difference and check if it's on the slice
        xx_prop = initial_theta*math.cos(phi) + nu*math.sin(phi)
        cur_lnpdf = lnpdf(xx_prop,*pdf_params)
        if cur_lnpdf > hh:
            # New point is on slice, ** EXIT LOOP **
            break
        # Shrink slice to rejected point
        if phi > 0:
            phi_max = phi
        elif phi < 0:
            phi_min = phi
        else:
            raise RuntimeError('BUG DETECTED: Shrunk to current position and still not acceptable.')
        # Propose new angle difference
        phi = np.random.uniform()*(phi_max - phi_min) + phi_min
    return (xx_prop,cur_lnpdf)


cov_K = np.zeros((Nfinal, Nfinal), dtype=np.float64)

rem=10
theta = alpha / beta
const=1
sigma0 = np.sqrt(theta) * const
g_mk_list2=[]
g_mk_list3=[]
theta_list=[]

for i in range(Nfinal):
        for j in range(i,Nfinal):
            if (j<Ngrid+N) and (i<Ngrid+N):
                cov_K[i][j]=expo_quad_kernel(xxx[i],xxx[j])
            if (j>(Ngrid+N-1)) and (i<(Ngrid+N)):
                cov_K[i][j]=expo_quad_kernel2(xxx[i],   days[j-Ngrid-N], days[j-Ngrid-N+1])
            if  (i>(Ngrid+N-1)):
                cov_K[i][j]=expo_quad_kernel3(days[i-Ngrid-N],days[i-Ngrid-N+1],  days[j-Ngrid-N],days[j-Ngrid-N+1 ]          )  
            if j!=i:
                cov_K[j][i]=cov_K[i][j]
# Construct ones_vec
ones_vec = np.ones(Nfinal)
ones_vec[Ngrid + N:] = diff
ones_vec = ones_vec.reshape(-1, 1)



# const is supplied as a command-line argument; sigma0 = sqrt(theta) * const
cov_K_corrected=cov_K+ones_vec@ones_vec.T*sigma0**2
cov_K_mod_chol = np.linalg.cholesky(cov_K_corrected)
cov_K_mod_inv=np.linalg.inv(cov_K_corrected)

curloglike = None
start_time=time.time()
for ite in range(nsim1+nsim2):
    if (ite%10000==0):
        print(ite)
    cov_K_chol_final=cov_K_mod_chol/np.sqrt(theta)
    prior=chol_sample(mean=np.zeros(Nfinal), cov_chol=cov_K_chol_final)#nu
    g_mk,curloglike=elliptical_slice(g_mk.reshape(1,-1),prior,log_lik,pdf_params=[N],cur_lnpdf=curloglike,angle_range=None)
  
    if (ite%rem==0):
        g_mk_list2.append(1/g_mk[0][0:Ngrid])
        g_mk_list3.append(1/g_mk[0][Ngrid:Ngrid+N])
        alpha_pos=alpha+Nfinal/2
        beta_pos=beta+1/2*g_mk[0]@cov_K_mod_inv@g_mk[0]
        theta=np.random.gamma(alpha_pos, 1/beta_pos,1)
        theta_list.append(theta)
timerun=time.time()-start_time
timerun_10000=timerun/(nsim2+nsim1)*10000

nn=int(nsim1/rem)       
#grids
low=np.quantile(np.array(g_mk_list2)[nn:,], 0.025, axis=0)
high=np.quantile(np.array(g_mk_list2)[nn:,], 0.975, axis=0)
med=np.quantile(np.array(g_mk_list2)[nn:,], 0.5, axis=0)

#np.savez('outputs/synthetic/boundedcoaldata/bounded/syn3/RI_BM/syn3_tips100_data_'+sys.argv[1]+'jitter7.npz', aaa=g_mk_list3,aa=g_mk_list2,c=points_inhomo,d=x4,e=truth,f=coverage1,i=jitter,j=l2_dist1,p=timerun_10000,q=width1,s=theta_list)

samples = np.array(g_mk_list2)[nn:]   # shape: n_samples × Ngrid

def hpd_interval(x, alpha=0.05):
    x = np.sort(x)
    n = len(x)
    k = int(np.floor((1 - alpha) * n))
    widths = x[k:] - x[:n-k]
    j = np.argmin(widths)
    return x[j], x[j+k]

low = np.zeros(samples.shape[1])
high = np.zeros(samples.shape[1])

for j in range(samples.shape[1]):
    low[j], high[j] = hpd_interval(samples[:, j], alpha=0.05)
    

fig, ax = plt.subplots()
#ax.plot(x4,truth,'k-',lw=3,alpha=0.6,label='Truth')
ax.plot(x4,med,'--',lw=2,alpha=0.6,c='r')
ax.plot(x4,low,'--',lw=2,alpha=0.6,c='r',label='SC-RI-BM')